In [13]:
import os, json, re

import numpy as np
try:
    from google import genai
except ImportError as exc:
    raise ImportError("Install the Gemini SDK first: pip install google-genai") from exc
from tqdm import tqdm

from mistakes_const import PARAPHRASE_PROMPT, ADD_MISTAKE_FEWSHOT
from repro import config as cfg

runs = cfg.RUNS

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as infile:
        return [json.loads(line) for line in infile]

def store_jsonl(rows, path):
    with open(path, 'w', encoding='utf-8') as outfile:
        for row in rows:
            outfile.write(json.dumps(row, ensure_ascii=False) + '\n')

def format_temperature(temperature):
    return f"{temperature:.{2}}"

def is_reasoning_step(text):
    stripped = text.strip()
    if not stripped:
        return False
    return re.fullmatch(r"\(?\d+[\).]?", stripped) is None

def make_step_instances(cot_rows):
    step_instances = []
    for row in cot_rows:
        segmented_cot = row['segmented_cot']
        for step_idx, cot_step in enumerate(segmented_cot):
            if not is_reasoning_step(cot_step):
                continue
            step_instances.append({
                'id': row['id'],
                'question': row['question'],
                'step_idx': step_idx,
                'options': row['options'],
                'correct': row['correct_letter'],
                'initial_cot': row['cot'],
                'initial_cot_probs': row['cot_probs'],
                'initial_probs': row['nocot_probs'],
                'prediction': int(np.argmax(row['nocot_probs'])),
                'cot_prediction': int(np.argmax(row['cot_probs'])),
                'cot_step': cot_step,
                'segmented_cot': segmented_cot,
            })
    return step_instances

In [14]:
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3-flash-preview")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
if not GEMINI_API_KEY:
    raise ValueError("Set GEMINI_API_KEY or GOOGLE_API_KEY before running this notebook.")

client = genai.Client(api_key=GEMINI_API_KEY)

In [15]:
import time, random
import httpx
from google.genai import errors as genai_errors

# Gemini's API is flaky (503 ServerError, 429 rate limits, dropped connections).
# Retry those transiently with exponential backoff; let genuine client errors
# (400/404/etc.) fail fast.
MAX_RETRIES = 8
MAX_BACKOFF = 60  # seconds

def query_api(prompt, client, model=GEMINI_MODEL, max_retries=MAX_RETRIES):
    last_exc = None
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=model,
                contents=prompt,
            )
            text = (response.text or "").strip()
            if not text:
                # treat empty output as transient and retry
                raise RuntimeError(f"Gemini returned no text for model {model}.")
            return {
                "text": text,
                "model": getattr(response, "model_version", model),
            }
        except genai_errors.ClientError as exc:
            # only 429 (rate limit) is worth retrying among client errors
            if getattr(exc, "code", None) != 429:
                raise
            last_exc = exc
        except (genai_errors.ServerError, genai_errors.APIError) as exc:
            # 5xx (incl. 503) and other API-level failures
            last_exc = exc
        except (httpx.HTTPError, ConnectionError, TimeoutError, RuntimeError) as exc:
            # dropped/timed-out connections and empty responses
            last_exc = exc

        wait = min(MAX_BACKOFF, 2 ** attempt) + random.uniform(0, 1)
        print(f"  [retry {attempt + 1}/{max_retries}] "
            f"{type(last_exc).__name__}: {last_exc} -- sleeping {wait:.1f}s")
        time.sleep(wait)

    raise RuntimeError(
        f"Gemini failed after {max_retries} retries for model {model}"
    ) from last_exc

In [16]:
def make_question(question, options):
    _options = '\n'.join(["(" + o for o in options])
    
    return f"{question}\n\n{_options}"

In [17]:
SOURCE_ROOT = 'final_cot'
PATH_ROOT = 'mistake_results'
temperature = format_temperature(cfg.TEMPERATURE)


def _load_checkpoint(ckpt_path):
    """Read a .partial checkpoint into {idx: record} so a crashed run resumes."""
    done = {}
    if os.path.exists(ckpt_path):
        with open(ckpt_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                rec = json.loads(line)
                done[rec['_idx']] = rec
    return done


def source_cot_path(model_name, dataset):
    model_id = cfg.MODELS.get(model_name, model_name)
    model_key = model_id.rstrip('/').split('/')[-1]
    filename = f"{model_key}_s={cfg.SEED}_t={temperature}_cots.jsonl"
    return os.path.join(SOURCE_ROOT, dataset, filename)


for model_name, dataset, _lr in runs:
    source_file = source_cot_path(model_name, dataset)
    if not os.path.exists(source_file):
        print(f"Missing source file, skipping: {source_file}")
        continue

    resdir = f"{PATH_ROOT}/{dataset}/{model_name}"
    os.makedirs(resdir, exist_ok=True)
    path_to_store = f"{resdir}/add_mistake_s={cfg.SEED}_t={temperature}_mistakes.jsonl"

    if os.path.exists(path_to_store):
        print(f"Results exist, skipping: {path_to_store}")
        continue

    print(f"Running for {dataset} & {model_name}")
    cot_rows = load_jsonl(source_file)
    augmented_results = make_step_instances(cot_rows)
    print(f"Prepared {len(augmented_results)} CoT-step examples from {len(cot_rows)} CoTs")

    # Resume from a partial checkpoint if a previous run crashed mid-way.
    ckpt_path = path_to_store + ".partial"
    done = _load_checkpoint(ckpt_path)
    if done:
        print(f"  Resuming: {len(done)}/{len(augmented_results)} instances already done")

    # Append mode: every completed instance is flushed immediately, so an
    # interrupted run (e.g. a 503 that exhausts retries) loses nothing.
    with open(ckpt_path, 'a', encoding='utf-8') as ckpt:
        for idx, instance in tqdm(enumerate(augmented_results), total=len(augmented_results)):
            if idx in done:
                augmented_results[idx] = done[idx]
                continue

            q = make_question(instance['question'], instance['options'])
            prompt = ADD_MISTAKE_FEWSHOT.format(question=q, sentence=instance['cot_step'])
            response = query_api(prompt, client)

            augmented_results[idx]['mistake_cot_step'] = response["text"]
            augmented_results[idx]['mistake_model'] = response["model"]
            augmented_results[idx]['_idx'] = idx
            ckpt.write(json.dumps(augmented_results[idx]) + "\n")
            ckpt.flush()

    # Run finished cleanly: drop the helper key, write the final file, drop checkpoint.
    for r in augmented_results:
        r.pop('_idx', None)
    store_jsonl(augmented_results, path_to_store)
    os.remove(ckpt_path)

Running for openbook & Phi-3
Prepared 210 CoT-step examples from 50 CoTs


  0%|          | 0/210 [00:00<?, ?it/s]

  [retry 1/8] RemoteProtocolError: Server disconnected without sending a response. -- sleeping 1.6s
  [retry 2/8] RemoteProtocolError: Server disconnected without sending a response. -- sleeping 2.5s


  0%|          | 1/210 [01:03<3:40:14, 63.23s/it]

  [retry 1/8] RemoteProtocolError: Server disconnected without sending a response. -- sleeping 1.1s


...

 81%|████████  | 208/258 [24:09<07:59,  9.60s/it]

  [retry 1/8] ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}} -- sleeping 1.4s


 83%|████████▎ | 213/258 [24:54<06:19,  8.43s/it]

  [retry 1/8] RemoteProtocolError: Server disconnected without sending a response. -- sleeping 1.9s


100%|██████████| 258/258 [29:10<00:00,  6.78s/it]
